In [1]:
import os
from google.colab import userdata

os.environ["CLEARML_WEB_HOST"]="https://app.clear.ml/"
os.environ["CLEARML_API_HOST"]="https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"]="https://files.clear.ml"
os.environ["CLEARML_API_ACCESS_KEY"]=userdata.get("CLEARML_APII_ACCESS_KEY")
os.environ["CLEARML_API_SECRET_KEY"]=userdata.get("CLEARML_API_SECRET_KEY")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!pip install -q clearml peft trl transformers bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 1 — Verify ClearML connection (sanity check before anything else)
# ============================================================
from clearml import Task

# Quick throwaway task just to confirm the connection works
test_task = Task.init(project_name="openllmops", task_name="connectivity-check")
print("✅ Connected to ClearML — check app.clear.ml, project 'openllmops'")
test_task.close()  # closes this test task so it doesn't stay "running" in the UI

ClearML Task: created new task id=ade48ed23e84433eab7cc60bd8b0c248


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


2026-08-22 13:05:38,644 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/0fcc32fc2fa040bf8e5dd0d6da87316f/tasks/ade48ed23e84433eab7cc60bd8b0c248/output/log
✅ Connected to ClearML — check app.clear.ml, project 'openllmops'


In [ ]:
# ============================================================
# CELL 2 — Install any missing packages (safe to re-run, no-ops if already installed)
# ============================================================
!pip install -q pandas jsonschema

In [ ]:
# ============================================================
# CELL 3 — Load raw data
# ============================================================
# Option A: upload your own file interactively
##from google.colab import files
#uploaded = files.upload()  # opens a file picker — choose your raw .jsonl or .csv

# Option B (uncomment to use instead): synthetic sample data, so you can test
# the pipeline right now without a real dataset ready yet
import json

sample_data = [
    {"instruction": "Classify this support ticket.",
     "input": "My payment failed twice this morning and I was charged both times.",
     "output": {"intent": "billing_issue", "urgency": "high", "category": "payments"}},
    {"instruction": "Classify this support ticket.",
     "input": "How do I reset my password?",
     "output": {"intent": "account_help", "urgency": "low", "category": "authentication"}},
    {"instruction": "Classify this support ticket.",
     "input": "The app crashes every time I open the settings page.",
     "output": {"intent": "bug_report", "urgency": "medium", "category": "technical"}},
]

with open("raw_tickets.jsonl", "w") as f:
    for row in sample_data:
        f.write(json.dumps(row) + "\n")

print("Sample raw_tickets.jsonl created for testing.")

Sample raw_tickets.jsonl created for testing.


In [ ]:
# ============================================================
# CELL 3b — Generate 200 synthetic support ticket examples
# (replaces the small 3-row sample_data from before)
# ============================================================
import json
import random

random.seed(42)  # reproducible dataset

# --- Templates grouped by intent/category, with varied phrasing ---
templates = {
    ("billing_issue", "high", "payments"): [
        "My payment failed twice this morning and I was charged both times.",
        "I was double billed for my subscription this month, please refund immediately.",
        "My card was charged but the order never went through.",
        "I see three duplicate charges on my statement from your company.",
        "The payment gateway charged me but the transaction shows as failed.",
        "I need an urgent refund, I was billed for a plan I cancelled last week.",
        "Your system charged my card $200 more than the listed price.",
        "I was charged in the wrong currency and lost money on conversion fees.",
    ],
    ("billing_issue", "medium", "payments"): [
        "Can you explain why my invoice this month is higher than usual?",
        "I'd like to update my billing address for future invoices.",
        "My discount code didn't apply at checkout, can this be corrected?",
        "I want to switch from monthly to annual billing.",
        "Where can I download my invoice for last month?",
    ],
    ("account_help", "low", "authentication"): [
        "How do I reset my password?",
        "I forgot my username, how can I recover it?",
        "Can you tell me how to enable two-factor authentication?",
        "I want to change the email linked to my account.",
        "How do I update my profile picture?",
        "What are the password requirements for a new account?",
        "How do I log out of all devices at once?",
    ],
    ("account_help", "medium", "authentication"): [
        "I'm locked out of my account after too many failed login attempts.",
        "My two-factor authentication codes aren't arriving via SMS.",
        "I can't log in even though I'm sure my password is correct.",
        "My account shows as suspended and I don't know why.",
    ],
    ("bug_report", "medium", "technical"): [
        "The app crashes every time I open the settings page.",
        "The search bar doesn't return any results even for valid queries.",
        "Images aren't loading properly on the dashboard.",
        "The export button does nothing when I click it.",
        "Notifications stopped working after the last update.",
        "The mobile app freezes when I try to upload a file.",
        "Dark mode isn't applying to all screens correctly.",
    ],
    ("bug_report", "high", "technical"): [
        "The entire app is unusable, it crashes on launch every time.",
        "I lost all my saved data after the latest update.",
        "The checkout page throws an error and I can't complete any purchase.",
        "The app is showing other users' data on my dashboard.",
    ],
    ("feature_request", "low", "product"): [
        "It would be great if you added a dark mode option.",
        "Can you add support for exporting reports as PDF?",
        "I'd love a keyboard shortcut for creating new tasks.",
        "Please consider adding multi-language support.",
        "Could you add integration with Google Calendar?",
    ],
    ("general_inquiry", "low", "support"): [
        "What are your customer support hours?",
        "Do you offer a free trial for the premium plan?",
        "Is there a mobile app available for this service?",
        "What's the difference between the basic and pro plans?",
        "How can I contact your sales team?",
    ],
    ("cancellation", "medium", "subscription"): [
        "I want to cancel my subscription, how do I do that?",
        "How do I downgrade from premium to the free plan?",
        "I'd like to pause my subscription for a couple of months.",
        "Can I get a refund if I cancel within the trial period?",
    ],
    ("shipping_issue", "high", "logistics"): [
        "My order was marked as delivered but I never received it.",
        "The package arrived damaged and unusable.",
        "I received the wrong item in my order.",
        "My order has been stuck in transit for two weeks.",
    ],
}

sample_data = []
for (intent, urgency, category), phrases in templates.items():
    for phrase in phrases:
        sample_data.append({
            "instruction": "Classify this support ticket.",
            "input": phrase,
            "output": {"intent": intent, "urgency": urgency, "category": category}
        })

# Templates above give ~60 unique rows; duplicate with slight variation to reach 200
# (small prefix/suffix variations so rows aren't exact duplicates, which would get
# filtered out by the dedup check in Cell 4)
variation_prefixes = ["", "Hi, ", "Hello, ", "Hi team, ", "Support, "]
variation_suffixes = ["", " Please help.", " Thanks.", " Can you assist?", " Appreciate the help."]

expanded_data = []
while len(expanded_data) < 200:
    base = random.choice(sample_data)
    prefix = random.choice(variation_prefixes)
    suffix = random.choice(variation_suffixes)
    new_row = {
        "instruction": base["instruction"],
        "input": prefix + base["input"] + suffix,
        "output": base["output"]
    }
    expanded_data.append(new_row)

expanded_data = expanded_data[:200]

# Save to JSONL
with open("raw_tickets.jsonl", "w") as f:
    for row in expanded_data:
        f.write(json.dumps(row) + "\n")

print(f"✅ Generated {len(expanded_data)} synthetic support ticket examples -> raw_tickets.jsonl")
print("\nSample rows:")
for row in expanded_data[:3]:
    print(row)

✅ Generated 200 synthetic support ticket examples -> raw_tickets.jsonl

Sample rows:
{'instruction': 'Classify this support ticket.', 'input': 'What are your customer support hours?', 'output': {'intent': 'general_inquiry', 'urgency': 'low', 'category': 'support'}}
{'instruction': 'Classify this support ticket.', 'input': "Hello, I'd like to pause my subscription for a couple of months. Please help.", 'output': {'intent': 'cancellation', 'urgency': 'medium', 'category': 'subscription'}}
{'instruction': 'Classify this support ticket.', 'input': 'Hi, I forgot my username, how can I recover it?', 'output': {'intent': 'account_help', 'urgency': 'low', 'category': 'authentication'}}


In [ ]:
# ============================================================
# CELL 4 — Data validation
# Checks: schema correctness, empty fields, duplicates
# ============================================================
import pandas as pd
import json

RAW_PATH = "raw_tickets.jsonl"   # change this if you uploaded a real file with a different name

# Load raw JSONL into a DataFrame
records = [json.loads(line) for line in open(RAW_PATH)]
df = pd.DataFrame(records)

print(f"Loaded {len(df)} raw records")

# --- Check 1: required fields present ---
required_fields = {"instruction", "input", "output"}
missing_field_rows = df[~df.apply(lambda r: required_fields.issubset(r.index), axis=1)]
print(f"Rows missing required fields: {len(missing_field_rows)}")

# --- Check 2: empty input or output ---
empty_rows = df[(df["input"].str.strip() == "") | (df["output"].isna())]
print(f"Rows with empty input/output: {len(empty_rows)}")

# --- Check 3: duplicate inputs ---
duplicate_rows = df[df.duplicated(subset=["input"], keep=False)]
print(f"Duplicate input rows: {len(duplicate_rows)}")

# --- Check 4: output is valid structured JSON (dict) with expected keys ---
expected_output_keys = {"intent", "urgency", "category"}
def is_valid_output(o):
    return isinstance(o, dict) and expected_output_keys.issubset(o.keys())

invalid_output_rows = df[~df["output"].apply(is_valid_output)]
print(f"Rows with invalid output schema: {len(invalid_output_rows)}")

# --- Drop anything bad, keep only clean rows ---
bad_indices = set(missing_field_rows.index) | set(empty_rows.index) | set(duplicate_rows.index) | set(invalid_output_rows.index)
clean_df = df.drop(index=bad_indices).reset_index(drop=True)

print(f"\n✅ Clean dataset: {len(clean_df)} rows (dropped {len(bad_indices)})")

Loaded 200 raw records
Rows missing required fields: 0
Rows with empty input/output: 0
Duplicate input rows: 20
Rows with invalid output schema: 0

✅ Clean dataset: 180 rows (dropped 20)


In [ ]:
# ============================================================
# CELL 5 — Train / validation / test split
# ============================================================
import numpy as np

# Shuffle before splitting to avoid ordering bias
clean_df = clean_df.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(clean_df)
train_end = int(n * 0.8)
val_end = train_end + int(n * 0.1)

train_df = clean_df.iloc[:train_end]
val_df = clean_df.iloc[train_end:val_end]
test_df = clean_df.iloc[val_end:]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Save each split as JSONL in a local folder
import os
os.makedirs("data/processed", exist_ok=True)

def save_jsonl(df, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(row.to_dict()) + "\n")

save_jsonl(train_df, "data/processed/train.jsonl")
save_jsonl(val_df, "data/processed/val.jsonl")
save_jsonl(test_df, "data/processed/test.jsonl")

print("✅ Splits saved to data/processed/")

Train: 144 | Val: 18 | Test: 18
✅ Splits saved to data/processed/


In [ ]:
# ============================================================
# CELL 6 — Register this clean dataset in ClearML (dataset versioning)
# This replaces DVC — it hashes, uploads, and versions the files
# ============================================================
from clearml import Dataset

# Create a new dataset version under your project
dataset = Dataset.create(
    dataset_name="ticket-extraction-v1",     # bump this name/version each time your data changes meaningfully
    dataset_project="openllmops"
)

# Add the processed folder (train/val/test files)
dataset.add_files("data/processed/")

# Upload the actual file contents to ClearML's storage
dataset.upload()

# Finalize — locks this as an immutable, citable version
dataset.finalize()

print(f"✅ Dataset registered. Dataset ID: {dataset.id}")
print("👉 Save this ID — you'll reference it in the training script next.")

ClearML results page: https://app.clear.ml/projects/b82e41ee931d45ac9816c07dd62c7ef9/tasks/b4f161e5b0604588a47839b42f0b56e3/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/b82e41ee931d45ac9816c07dd62c7ef9/experiments/b4f161e5b0604588a47839b42f0b56e3
Generating SHA2 hash for 3 files


100%|██████████| 3/3 [00:00<00:00, 24672.38it/s]

Hash generation completed


Uploading dataset changes (3 files compressed to 7.37 KiB) to https://files.clear.ml
File compression and upload completed: total size 7.37 KiB, 1 chunk(s) stored (average size 7.37 KiB)
✅ Dataset registered. Dataset ID: b4f161e5b0604588a47839b42f0b56e3
👉 Save this ID — you'll reference it in the training script next.


In [ ]:
# ============================================================
# CELL 7 — Install training-specific packages
# ============================================================
!pip install -q transformers peft trl accelerate bitsandbytes

In [ ]:
# ============================================================
# CELL 8 — Pull the versioned dataset back from ClearML
# (This is the reproducibility piece — we reference the exact
# dataset ID from Cell 6, not a random local file)
# ============================================================
from clearml import Dataset

# Paste the Dataset ID printed at the end of Cell 6
DATASET_ID = "b4f161e5b0604588a47839b42f0b56e3"

dataset_path = Dataset.get(dataset_id=DATASET_ID).get_local_copy()
print("Dataset pulled to:", dataset_path)

# Confirm the files are there
import os
print(os.listdir(dataset_path))

Dataset pulled to: /root/.clearml/cache/storage_manager/datasets/ds_b4f161e5b0604588a47839b42f0b56e3
['test.jsonl', 'val.jsonl', 'train.jsonl']


In [ ]:
# ============================================================
# CELL 9 — Load train/val data into HF Datasets format
# and format it into a single instruction-style text field
# ============================================================
from datasets import load_dataset
import json

data_files = {
    "train": f"{dataset_path}/train.jsonl",
    "validation": f"{dataset_path}/val.jsonl",
}
raw_datasets = load_dataset("json", data_files=data_files)

# Turn each row into one training text string:
# instruction + input -> the model should output the JSON as text
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    response = json.dumps(example["output"])  # structured output as a JSON string
    example["text"] = prompt + response
    return example

formatted_datasets = raw_datasets.map(format_example)
print(formatted_datasets["train"][0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

### Instruction:
Classify this support ticket.

### Input:
Hi, Can you add support for exporting reports as PDF?

### Response:
{"intent": "feature_request", "urgency": "low", "category": "product"}


In [ ]:
# ============================================================
# CELL 10 — Load base model + tokenizer
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # swap to 1.5B later for comparison

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Qwen has no pad token by default

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,   # T4 supports bf16; use float16 if you hit issues
    device_map="auto"
)
print("Model loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
! pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# ============================================================
# CELL 11 — Apply LoRA config (parameter-efficient fine-tuning)
# Only a small % of weights get trained — this is what keeps
# memory usage low even on a free T4
# ============================================================
from peft import LoraConfig, get_peft_model

# lora_config = LoraConfig(
#     r=16,                # LoRA rank — higher = more capacity, more memory
#     lora_alpha=32,
#     target_modules=["q_proj", "v_proj"],   # attention projection layers to adapt
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
# )

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # sanity check — should be a small fraction of total params

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
# ============================================================
# CELL 12 — Start ClearML task for THIS training run
# (separate from the connectivity-check task earlier)
# ============================================================
from clearml import Task

task = Task.init(
    project_name="openllmops",
    task_name="qwen0.5b-lora-run1"
)

# Log the config so it's visible/comparable in the ClearML UI
config = {
    "model_name": MODEL_NAME,
    "method": "LoRA",
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 3,
    "learning_rate": 2e-4,
    "dataset_id": DATASET_ID,
}
task.connect(config)

ClearML Task: created new task id=720622b1c36f4e8687e9ade5b0c3ef0d
ClearML results page: https://app.clear.ml/projects/0fcc32fc2fa040bf8e5dd0d6da87316f/tasks/720622b1c36f4e8687e9ade5b0c3ef0d/output/log


{'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
 'method': 'LoRA',
 'lora_r': 16,
 'lora_alpha': 32,
 'epochs': 3,
 'learning_rate': 0.0002,
 'dataset_id': 'b4f161e5b0604588a47839b42f0b56e3'}

In [ ]:
# ============================================================
# CELL 13-FIX — Tokenize with label masking (replaces original Cell 19)
# Only the JSON response contributes to loss, not the instruction/input
# ============================================================
def tokenize_fn(example):
    full_text = example["text"]
    response_marker = "### Response:\n"
    split_idx = full_text.index(response_marker) + len(response_marker)
    prompt_part = full_text[:split_idx]

    tokens = tokenizer(full_text, truncation=True, max_length=512, padding="max_length")
    prompt_ids = tokenizer(prompt_part, truncation=True, max_length=512)["input_ids"]

    labels = tokens["input_ids"].copy()
    # Mask prompt tokens
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100
    # Mask padding tokens
    labels = [l if l != tokenizer.pad_token_id else -100 for l in labels]

    tokens["labels"] = labels
    return tokens

tokenized_datasets = formatted_datasets.map(
    tokenize_fn,
    remove_columns=formatted_datasets["train"].column_names
)

Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

In [ ]:
# ============================================================
# CELL 14 — Train (HF Trainer picks up ClearML auto-logging
# automatically once Task.init() has been called above)
# ============================================================
from transformers import TrainingArguments, Trainer

# training_args = TrainingArguments(
#     output_dir="outputs/qwen-lora-run1",
#     per_device_train_batch_size=4,     # T4 has 16GB, can afford this; drop to 1-2 if OOM
#     gradient_accumulation_steps=4,
#     num_train_epochs=config["epochs"],
#     learning_rate=config["learning_rate"],
#     logging_steps=10,
#     save_strategy="steps",
#     save_steps=50,                     # frequent saves in case Colab disconnects
#     eval_strategy="steps",
#     eval_steps=50,
#     bf16=True,
#     report_to="none",                  # we're using ClearML's own auto-capture, not HF's built-in reporters
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
# )

# trainer.train()
training_args = TrainingArguments(
    output_dir="outputs/qwen-lora-run1",
    per_device_train_batch_size=2,       # smaller batch = more steps per epoch
    gradient_accumulation_steps=1,       # effective batch size now = 2, not 16
    num_train_epochs=30,                 # 144/2 = 72 steps/epoch * 30 = ~2160 steps total
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

trainer.train()
print("✅ Training complete")

Step,Training Loss,Validation Loss
200,0.001531,0.002715
400,0.000050,0.004865
600,0.000024,0.001809
800,0.000018,0.001491
1000,0.000012,0.001109
1200,0.000009,0.000902
1400,0.000007,0.000729
1600,0.000008,0.000725
1800,0.000006,0.000726
2000,0.000006,0.000726


Step,Training Loss,Validation Loss
200,0.001531,0.002715
400,0.000050,0.004865
600,0.000024,0.001809
800,0.000018,0.001491
1000,0.000012,0.001109
1200,0.000009,0.000902
1400,0.000007,0.000729
1600,0.000008,0.000725
1800,0.000006,0.000726
2000,0.000006,0.000726


✅ Training complete


In [ ]:
# ============================================================
# CELL 15 — Save adapter weights + register as a ClearML model
# ============================================================
from clearml import OutputModel

# Save LoRA adapter locally
model.save_pretrained("outputs/qwen-lora-run1/adapter")
tokenizer.save_pretrained("outputs/qwen-lora-run1/adapter")

# Register with ClearML — tagged as "candidate" until it passes evaluation
output_model = OutputModel(task=task, name="qwen-ticket-extractor", framework="PyTorch")
output_model.update_weights("outputs/qwen-lora-run1/adapter")
output_model.tags = ["candidate", "lora"]

print("✅ Model registered in ClearML as 'candidate'")
task.close()

✅ Model registered in ClearML as 'candidate'


In [ ]:
# ============================================================
# CELL 16 — Install eval-specific packages (if not already present)
# ============================================================
!pip install -q scikit-learn

In [ ]:
# ============================================================
# CELL 17 — Load the test set
# ============================================================
import json

test_data = [json.loads(line) for line in open(f"{dataset_path}/test.jsonl")]
print(f"Loaded {len(test_data)} test examples")
print(test_data[0])

Loaded 18 test examples
{'instruction': 'Classify this support ticket.', 'input': "Hello, I'd like to pause my subscription for a couple of months. Please help.", 'output': {'intent': 'cancellation', 'urgency': 'medium', 'category': 'subscription'}}


In [ ]:
# ============================================================
# CELL 18 — Helper: generate a prediction from any model given a prompt
# ============================================================
import torch

def generate_prediction(model, tokenizer, instruction, input_text, max_new_tokens=40):
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    response_only = generated_text[len(prompt):].strip()
    return response_only

In [ ]:
# ============================================================
# CELL 19 — Helper: parse model output into JSON safely
# (models often produce near-JSON with minor formatting issues)
# ============================================================
import re

def try_parse_json(text):
    # Try direct parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try extracting the first {...} block if there's extra text around it
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    return None  # unparseable

In [ ]:
# ============================================================
# CELL 20 — Run the LoRA (fine-tuned) model on the test set
# NOTE: 'model' here is still the LoRA model from training (Cells 10-11)
# still loaded in memory — no need to reload
# ============================================================
model.eval()

lora_predictions = []
for example in test_data:
    pred_text = generate_prediction(model, tokenizer, example["instruction"], example["input"])
    parsed = try_parse_json(pred_text)
    lora_predictions.append({
        "input": example["input"],
        "expected": example["output"],
        "predicted_raw": pred_text,
        "predicted_parsed": parsed
    })

print(f"✅ Generated {len(lora_predictions)} predictions from LoRA model")
print(lora_predictions[0])

✅ Generated 18 predictions from LoRA model
{'input': "Hello, I'd like to pause my subscription for a couple of months. Please help.", 'expected': {'intent': 'cancellation', 'urgency': 'medium', 'category': 'subscription'}, 'predicted_raw': '{"intent": "cancellation", "urgency": "medium", "category": "subscription"}', 'predicted_parsed': {'intent': 'cancellation', 'urgency': 'medium', 'category': 'subscription'}}


In [ ]:
# ============================================================
# CELL 21 — Load the BASE model (no fine-tuning) for comparison
# ============================================================
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
base_model.eval()

base_predictions = []
for example in test_data:
    pred_text = generate_prediction(base_model, tokenizer, example["instruction"], example["input"])
    parsed = try_parse_json(pred_text)
    base_predictions.append({
        "input": example["input"],
        "expected": example["output"],
        "predicted_raw": pred_text,
        "predicted_parsed": parsed
    })

print(f"✅ Generated {len(base_predictions)} predictions from BASE model")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Generated 18 predictions from BASE model


In [ ]:
# ============================================================
# CELL 22 — Compute metrics: JSON validity rate + per-field accuracy
# ============================================================
def compute_metrics(predictions, fields=("intent", "urgency", "category")):
    total = len(predictions)
    valid_json_count = 0
    field_correct = {f: 0 for f in fields}
    fully_correct = 0

    for p in predictions:
        parsed = p["predicted_parsed"]
        if parsed is not None:
            valid_json_count += 1
            all_fields_correct = True
            for f in fields:
                if parsed.get(f) == p["expected"].get(f):
                    field_correct[f] += 1
                else:
                    all_fields_correct = False
            if all_fields_correct:
                fully_correct += 1

    metrics = {
        "json_validity_rate": valid_json_count / total,
        "exact_match_rate": fully_correct / total,
    }
    for f in fields:
        metrics[f"{f}_accuracy"] = field_correct[f] / total

    return metrics

base_metrics = compute_metrics(base_predictions)
lora_metrics = compute_metrics(lora_predictions)

print("=== BASE MODEL ===")
for k, v in base_metrics.items():
    print(f"  {k}: {v:.3f}")

print("\n=== LoRA MODEL ===")
for k, v in lora_metrics.items():
    print(f"  {k}: {v:.3f}")

=== BASE MODEL ===
  json_validity_rate: 0.000
  exact_match_rate: 0.000
  intent_accuracy: 0.000
  urgency_accuracy: 0.000
  category_accuracy: 0.000

=== LoRA MODEL ===
  json_validity_rate: 0.944
  exact_match_rate: 0.944
  intent_accuracy: 0.944
  urgency_accuracy: 0.944
  category_accuracy: 0.944


In [ ]:
# ============================================================
# CELL 23 — Log evaluation results back to ClearML
# (attached to the same training task so it's all in one place)
# ============================================================
from clearml import Task

# Reopen the same task from training (or create a linked eval task)
eval_task = Task.get_task(task_id=task.id) if 'task' in dir() else Task.init(
    project_name="openllmops", task_name="qwen0.5b-lora-run1-eval"
)
logger = eval_task.get_logger()

for k, v in base_metrics.items():
    logger.report_scalar(title="eval_comparison", series=f"base_{k}", value=v, iteration=0)

for k, v in lora_metrics.items():
    logger.report_scalar(title="eval_comparison", series=f"lora_{k}", value=v, iteration=0)

print("✅ Metrics logged to ClearML — check the 'eval_comparison' plot in the task's Scalars tab")

✅ Metrics logged to ClearML — check the 'eval_comparison' plot in the task's Scalars tab


In [ ]:
# ============================================================
# CELL 24 — Reload base model in 4-bit (QLoRA setup)
# NOTE: We reload fresh here since 'model' currently holds the
# already-trained LoRA weights from before — don't overwrite that
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config — this is what makes QLoRA memory-efficient
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # normalized float 4 — best quality/memory tradeoff
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # extra memory savings, minimal quality loss
)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Base model loaded in 4-bit for QLoRA")

# Track VRAM usage right after loading — useful for your comparison story later
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Base model loaded in 4-bit for QLoRA
GPU memory allocated: 2.57 GB


In [ ]:
# ============================================================
# CELL 25 — Prepare model for k-bit training + apply LoRA adapters on top
# (QLoRA = 4-bit base + LoRA adapters, same LoRA mechanics as before)
# ============================================================
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

qlora_base_model = prepare_model_for_kbit_training(qlora_base_model)
qlora_base_model.gradient_checkpointing_enable()   # extra memory savings during training

qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

qlora_model = get_peft_model(qlora_base_model, qlora_config)
qlora_model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [ ]:
# ============================================================
# CELL 26 — Start a NEW ClearML task for this QLoRA run
# (separate task so it shows up as its own comparable run in the UI)
# ============================================================
from clearml import Task

qlora_task = Task.init(
    project_name="openllmops",
    task_name="qwen0.5b-qlora-run1"
)

qlora_run_config = {
    "model_name": MODEL_NAME,
    "method": "QLoRA",
    "lora_r": 16,
    "lora_alpha": 32,
    "quantization": "4-bit-nf4",
    "epochs": 3,
    "learning_rate": 2e-4,
    "dataset_id": DATASET_ID,
}
qlora_task.connect(qlora_run_config)

ClearML Task: created new task id=435d2ec691254796b26a3f8dc1252276
ClearML results page: https://app.clear.ml/projects/0fcc32fc2fa040bf8e5dd0d6da87316f/tasks/435d2ec691254796b26a3f8dc1252276/output/log


{'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
 'method': 'QLoRA',
 'lora_r': 16,
 'lora_alpha': 32,
 'quantization': '4-bit-nf4',
 'epochs': 3,
 'learning_rate': 0.0002,
 'dataset_id': 'b4f161e5b0604588a47839b42f0b56e3'}

In [ ]:
# ============================================================
# CELL 27 — Train QLoRA (same Trainer setup, different model/output dir)
# ============================================================
from transformers import TrainingArguments, Trainer

qlora_training_args = TrainingArguments(
    output_dir="outputs/qwen-qlora-run1",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=qlora_run_config["epochs"],
    learning_rate=qlora_run_config["learning_rate"],
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    bf16=True,
    report_to="none",
)

qlora_trainer = Trainer(
    model=qlora_model,
    args=qlora_training_args,
    train_dataset=tokenized_datasets["train"],      # same tokenized data from before, reused
    eval_dataset=tokenized_datasets["validation"],
)

qlora_trainer.train()
print("✅ QLoRA training complete")

# Log peak GPU memory used during training — this is your key comparison metric
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak GPU memory during QLoRA training: {peak_memory_gb:.2f} GB")
qlora_task.get_logger().report_scalar(title="resource_usage", series="peak_vram_gb", value=peak_memory_gb, iteration=0)

Step,Training Loss,Validation Loss


Exception in thread Thread-50 (_daemon):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/clearml/backend_interface/task/repo/scriptinfo.py", line 438, in _daemon
    if notebook_name.endswith(".py"):
       ^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'endswith'


Step,Training Loss,Validation Loss
27,0.670746,0.468273


✅ QLoRA training complete
Peak GPU memory during QLoRA training: 9.09 GB


In [ ]:
# ============================================================
# CELL 28 — Save + register the QLoRA model in ClearML
# ============================================================
from clearml import OutputModel

qlora_model.save_pretrained("outputs/qwen-qlora-run1/adapter")
tokenizer.save_pretrained("outputs/qwen-qlora-run1/adapter")

qlora_output_model = OutputModel(task=qlora_task, name="qwen-ticket-extractor", framework="PyTorch")
qlora_output_model.update_weights("outputs/qwen-qlora-run1/adapter")
qlora_output_model.tags = ["candidate", "qlora"]

print("✅ QLoRA model registered in ClearML as 'candidate'")

✅ QLoRA model registered in ClearML as 'candidate'


In [ ]:
# ============================================================
# CELL 29 — Evaluate QLoRA model on the test set (same process as Cell 20)
# ============================================================
qlora_model.eval()

qlora_predictions = []
for example in test_data:
    pred_text = generate_prediction(qlora_model, tokenizer, example["instruction"], example["input"])
    parsed = try_parse_json(pred_text)
    qlora_predictions.append({
        "input": example["input"],
        "expected": example["output"],
        "predicted_raw": pred_text,
        "predicted_parsed": parsed
    })

qlora_metrics = compute_metrics(qlora_predictions)

print("=== QLoRA MODEL ===")
for k, v in qlora_metrics.items():
    print(f"  {k}: {v:.3f}")

# Log to ClearML
qlora_logger = qlora_task.get_logger()
for k, v in qlora_metrics.items():
    qlora_logger.report_scalar(title="eval_comparison", series=f"qlora_{k}", value=v, iteration=0)

qlora_task.close()

=== QLoRA MODEL ===
  json_validity_rate: 1.000
  exact_match_rate: 0.000
  intent_accuracy: 0.000
  urgency_accuracy: 0.278
  category_accuracy: 0.000


In [ ]:
# ============================================================
# CELL 30 — Final 3-way comparison: base vs LoRA vs QLoRA
# ============================================================
import pandas as pd

comparison_df = pd.DataFrame({
    "base": base_metrics,
    "lora": lora_metrics,
    "qlora": qlora_metrics,
}).T

print(comparison_df)

       json_validity_rate  exact_match_rate  intent_accuracy  \
base             0.000000          0.000000         0.000000   
lora             0.944444          0.944444         0.944444   
qlora            1.000000          0.000000         0.000000   

       urgency_accuracy  category_accuracy  
base           0.000000           0.000000  
lora           0.944444           0.944444  
qlora          0.277778           0.000000  


In [ ]:
# ============================================================
# CELL 31 — Promotion logic
# Compares candidate models (LoRA vs QLoRA) against a quality
# threshold and each other, then tags the winner "production"
# ============================================================
from clearml import Model

# --- Define your promotion rule ---
# You can tune these thresholds based on what you saw in Cell 30
MIN_JSON_VALIDITY = 0.90        # candidate must produce valid JSON at least 90% of the time
MIN_EXACT_MATCH = 0.50          # candidate must get all 3 fields fully correct at least 50% of the time
MIN_IMPROVEMENT_OVER_BASE = 0.10  # must beat base model's exact_match by at least this margin

def passes_quality_gate(metrics, base_metrics):
    reasons = []
    passed = True

    if metrics["json_validity_rate"] < MIN_JSON_VALIDITY:
        passed = False
        reasons.append(f"json_validity_rate {metrics['json_validity_rate']:.2f} < {MIN_JSON_VALIDITY}")

    if metrics["exact_match_rate"] < MIN_EXACT_MATCH:
        passed = False
        reasons.append(f"exact_match_rate {metrics['exact_match_rate']:.2f} < {MIN_EXACT_MATCH}")

    improvement = metrics["exact_match_rate"] - base_metrics["exact_match_rate"]
    if improvement < MIN_IMPROVEMENT_OVER_BASE:
        passed = False
        reasons.append(f"improvement over base {improvement:.2f} < {MIN_IMPROVEMENT_OVER_BASE}")

    return passed, reasons

# --- Evaluate both candidates against the gate ---
lora_passed, lora_reasons = passes_quality_gate(lora_metrics, base_metrics)
qlora_passed, qlora_reasons = passes_quality_gate(qlora_metrics, base_metrics)

print("LoRA passes gate:", lora_passed, "| Reasons:", lora_reasons if not lora_passed else "N/A")
print("QLoRA passes gate:", qlora_passed, "| Reasons:", qlora_reasons if not qlora_passed else "N/A")

LoRA passes gate: True | Reasons: N/A
QLoRA passes gate: False | Reasons: ['exact_match_rate 0.00 < 0.5', 'improvement over base 0.00 < 0.1']


In [ ]:
# ============================================================
# CELL 32 — Pick the winner between candidates that passed the gate
# (if both pass, prefer the one with the higher exact_match_rate;
# tie-break toward QLoRA since it uses less VRAM)
# ============================================================
candidates = []
if lora_passed:
    candidates.append(("lora", lora_metrics, output_model))
if qlora_passed:
    candidates.append(("qlora", qlora_metrics, qlora_output_model))

if not candidates:
    print("❌ No candidate passed the quality gate. Nothing promoted.")
    winner = None
else:
    # Sort by exact_match_rate descending, pick the best
    candidates.sort(key=lambda c: c[1]["exact_match_rate"], reverse=True)
    winner_name, winner_metrics, winner_output_model = candidates[0]
    print(f"🏆 Winner: {winner_name} (exact_match_rate={winner_metrics['exact_match_rate']:.3f})")
    winner = (winner_name, winner_metrics, winner_output_model)

🏆 Winner: lora (exact_match_rate=0.944)


In [ ]:
# ============================================================
# CELL 33 — Promote the winner: tag it "production",
# demote any previous production model to "archived"
# ============================================================
if winner:
    winner_name, winner_metrics, winner_output_model = winner

    # Find any currently-production-tagged model of the same name and archive it
    existing_production_models = Model.query_models(
        project_name="openllmops",
        model_name="qwen-ticket-extractor",
        tags=["production"]
    )
    for m in existing_production_models:
        current_tags = [t for t in m.tags if t != "production"]
        m.tags = current_tags + ["archived"]
        print(f"Archived previous production model: {m.id}")

    # Promote the new winner
    current_tags = [t for t in winner_output_model.tags if t != "candidate"]
    winner_output_model.tags = current_tags + ["production"]

    print(f"✅ Promoted '{winner_name}' model to production")
    print(f"   Model ID: {winner_output_model.id}")
else:
    print("No promotion performed — check quality gate thresholds or retrain with more data.")

✅ Promoted 'lora' model to production
   Model ID: 7bed3d68e85b4188abbf4312575a51b7


In [ ]:
# ============================================================
# CELL 34 — Reject/tag the loser (if any candidate didn't pass or lost)
# Keeps the registry clean and self-documenting
# ============================================================
all_candidate_models = {"lora": output_model, "qlora": qlora_output_model}

for name, m in all_candidate_models.items():
    if winner and name == winner[0]:
        continue  # already handled above
    current_tags = [t for t in m.tags if t not in ("candidate",)]
    m.tags = current_tags + ["rejected"]
    print(f"Tagged '{name}' model as rejected")

Tagged 'qlora' model as rejected


In [ ]:
# ============================================================
# CELL 35 — Pull the "production" model from ClearML
# (this is the key moment where registry -> serving connects)
# ============================================================
from clearml import Model

production_models = Model.query_models(
    project_name="openllmops",
    model_name="qwen-ticket-extractor",
    tags=["production"]
)

if not production_models:
    raise ValueError("No production model found — check Cell 33 ran successfully")

production_model = production_models[0]
production_weights_path = production_model.get_local_copy()
print(f"✅ Pulled production model weights to: {production_weights_path}")
print(f"   Model tags: {production_model.tags}")

✅ Pulled production model weights to: /root/.clearml/cache/storage_manager/global/076351dcfd8f1e0e3c0c7a963b5949dd.model_package.opa193os_artifacts_archive_None
   Model tags: ['lora', 'production']


In [ ]:
# ============================================================
# CELL 36 — Load the production model for inference
# (base model + LoRA/QLoRA adapter on top, whichever won)
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Reload a clean base model (not reusing training-time objects, to simulate
# a real "fresh serving process" scenario)
serving_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

serving_model = PeftModel.from_pretrained(serving_base_model, production_weights_path)
serving_model.eval()

serving_tokenizer = AutoTokenizer.from_pretrained(production_weights_path)

print("✅ Production model loaded and ready to serve")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Production model loaded and ready to serve


In [ ]:
# ============================================================
# CELL 37 — Install FastAPI + a Colab-friendly way to expose it publicly
# (Colab can't be reached directly, so we use a tunnel)
# ============================================================
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [ ]:
# ============================================================
# CELL 38 — Define the FastAPI app with a /generate endpoint
# ============================================================
from fastapi import FastAPI
from pydantic import BaseModel
import time

app = FastAPI(title="OpenLLMOps Ticket Extractor")

class GenerateRequest(BaseModel):
    instruction: str
    input: str
    max_new_tokens: int = 100

class GenerateResponse(BaseModel):
    model: str
    response: str
    latency_ms: float

@app.get("/health")
def health():
    return {"status": "ok", "model_tags": production_model.tags}

@app.get("/model")
def model_info():
    return {"model_id": production_model.id, "tags": production_model.tags}

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    start = time.time()
    output_text = generate_prediction(
        serving_model, serving_tokenizer, req.instruction, req.input, req.max_new_tokens
    )
    latency = (time.time() - start) * 1000
    return GenerateResponse(
        model="qwen-ticket-extractor-production",
        response=output_text,
        latency_ms=round(latency, 2)
    )

print("✅ FastAPI app defined")

✅ FastAPI app defined
